# Eksplorasi AutoModelForSeq2SeqLM

**Task**: Conditional Generation (Penerjemahan Bahasa, Peringkas Teks / Summarization)
**Cara Kerja**: Menggunakan arsitektur Encoder-Decoder. Ia menerima satu *sequence* penuh (input dikodekan oleh Encoder), lalu menghasilkan *sequence* teks yang benar-benar baru secara bertahap (dihasilkan oleh Decoder).
**Model Populer**: T5, BART, Pegasus, MarianMT.
**Dataset**: `cnn_dailymail` - dataset populer berisi artikel berita panjang dan ringkasan utamanya (highlights). Sangat umum dipakai melatih model peringkas teks.

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
from datasets import load_dataset
import torch

## 1. Load Dataset Publik (`cnn_dailymail`)
Dataset ini memiliki dua kolom penting: `article` (teks panjang asli) dan `highlights` (hasil ringkasan/target dari model seq2seq).

In [5]:
dataset = load_dataset("cnn_dailymail", "3.0.0", split="train")

print("--- Contoh Data Index-0 ---")
article = dataset[0]["article"]
summary = dataset[0]["highlights"]
id = dataset[0]["id"]

print(f"ID: {id}")
print(f"ARTIKEL (Input untuk Encoder):\n{article[:400]}... [dipotong]\n")
print(f"RINGKASAN (Target Output Decoder):\n{summary}")

--- Contoh Data Index-0 ---
ID: 42c027e4ff9730fbb3de84c1af0d2c506e41c3e4
ARTIKEL (Input untuk Encoder):
LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappointment of gossip columnists around the world, the young actor says he has no plans to fritter his ca... [dipotong]

RINGKASAN (Target Output Decoder):
Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .


## 2. Load Tokenizer & Model
Saya akan menggunakan `google-t5/t5-small`, salah satu varian dari T5 (Text-to-Text Transfer Transformer). T5 unik karena bisa melakukan banyak tugas sekaligus, tetapi ia mewajibkan kita memberikan "prefix" sebagai instruksi tugas, misalnya: `"summarize: {teks}"` atau `"translate English to German: {teks}"`.

In [3]:
model_checkpoint = "google-t5/t5-small"
# Terkadang t5 membutuhkan use_fast=False bergantung versi environment, tapi kita pakai standar dulu
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Pastikan menggunakan AutoModelForSeq2SeqLM
model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [7]:
print(model)

T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

## 3. Inferensi Manual (Summarization)
Pada Seq2Seq, model tidak menggunakan argmax langsung pada logits untuk satu prediksi. Ia harus memanggil metodologi *generation* seperti pada model Causal LM, karena sifat Decoder adalah AutoRegressive.

In [6]:
text_to_summarize = """
The World Health Organization (WHO) is a specialized agency of the United Nations responsible for international public health. 
The WHO Constitution, which establishes the agency's governing structure and principles, states its main objective as "the attainment by all peoples of the highest possible level of health". 
It is headquartered in Geneva, Switzerland, with six semi-autonomous regional offices and 150 field offices worldwide.
"""

# T5 memerlukan prefix tugas.
input_text = "summarize: " + text_to_summarize

inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True)

# Menggunakan generate di model seq2seq
outputs = model.generate(
    **inputs,
    max_new_tokens=50,   # Panjang maksimum ringkasan yang dihasilkan
    min_length=10,       # Panjang minimun agar kalimat lebih bermakna
    length_penalty=2.0,  # Memaksa model membangkitkan teks yang tidak terlalu pendek atau terlalu panjang
    num_beams=4,         # Menggunakan Beam Search agar output teks yang digenerate lebih logis
    early_stopping=True
)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Hasil Prediksi Manual (Summarization) ---")
print("Ringkasan:\n", summary)

--- Hasil Prediksi Manual (Summarization) ---
Ringkasan:
 the world health organization (WHO) is a specialized agency of the united nations. it is headquartered in Geneva, Switzerland, with six semi-autonomous regional offices.


## 4. Inferensi Manual (Translation)
Model T5 juga dilatih untuk menerjemahkan teks di model basisnya.

In [10]:
# Menggunakan instruksi terjemahan
input_text_translate = "translate English to French: The World Health Organization is headquartered in Geneva, Switzerland."

inputs_trans = tokenizer(input_text_translate, return_tensors="pt")
outputs_trans = model.generate(**inputs_trans, max_new_tokens=40)
translated_text = tokenizer.decode(outputs_trans[0], skip_special_tokens=True)

print("--- Hasil Prediksi Manual (Translation) ---")
print("Terjemahan:\n", translated_text)

--- Hasil Prediksi Manual (Translation) ---
Terjemahan:
 L'Organisation mondiale de la santé a son siège à Genève (Suisse).


## 5. Persiapan Data untuk PyTorch Training Mandiri (Seq2Seq)
Pada task Sequence-to-Sequence (seperti peringkas teks/Summarization), dataset harus dibagi dua: teks input untuk **Encoder** dan teks target untuk **Decoder**. 

Di Hugging Face, kita mem-passing teks encoder ke memori `input_ids` biasa, sedangkan teks target ringkasan kita gunakan `tokenizer(text_target=...)` untuk mengisikan parameter `labels`. Nilai `-100` pada label digunakan agar PyTorch mengabaikan nilai padding saat menghitung error Loss.

In [11]:
from torch.utils.data import Dataset, DataLoader

# Ambil sampel kecil untuk simulasi (agar laptop/komputer gampang menghitungnya)
train_sample = dataset.select(range(50))
prefix = "summarize: "

def preprocess_function(examples):
    # 1. Tambahkan prefix di semua teks input Encoder
    inputs = [prefix + doc for doc in examples["article"]]
    
    # 2. Tokenisasi input (untuk Encoder)
    model_inputs = tokenizer(inputs, max_length=256, truncation=True, padding="max_length")

    # 3. Tokenisasi target (untuk Decoder) menggunakan text_target
    labels = tokenizer(text_target=examples["highlights"], max_length=64, truncation=True, padding="max_length")

    # 4. Ganti padding token di labels dengan -100 agar diabaikan dalam kalkulasi Loss CrossEntropy PyTorch
    labels["input_ids"] = [
        [(l if l != tokenizer.pad_token_id else -100) for l in label] for label in labels["input_ids"]
    ]

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Eksekusi mapping pre-process data
tokenized_train = train_sample.map(preprocess_function, batched=True, remove_columns=dataset.column_names)

class Seq2SeqDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            "input_ids": torch.tensor(item["input_ids"]),
            "attention_mask": torch.tensor(item["attention_mask"]),
            "labels": torch.tensor(item["labels"])
        }

train_dataloader = DataLoader(Seq2SeqDataset(tokenized_train), batch_size=4, shuffle=True)
print(f"Total Batch Seq2Seq: {len(train_dataloader)}")
print("Selesai memecah data teks Encoder dan target Decoder!")

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

Total Batch Seq2Seq: 13
Selesai memecah data teks Encoder dan target Decoder!


## 6. Proses PyTorch Training Loop
Loop training Seq2Seq cukup rapi karena arsitektur Hugging Face `AutoModelForSeq2SeqLM` mengambil alih kerumitan *teacher forcing*. Saat kita passing `labels` ke dalam *forward pass*, model otomatis menggeser tensor label ke kanan satu spasi untuk digunakan sebagai `decoder_input_ids`! Proses *loss.backward()* berjalan mulus seketika.

In [12]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
optimizer = AdamW(model.parameters(), lr=5e-5)

epochs = 1
print("==== Memulai Training Seq2Seq (Summarization) ===")
model.train()
for epoch in range(epochs):
    total_train_loss = 0
    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Loss untuk Autoregressive Decoder yang dikondisikan oleh pemahaman Encoder
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 5 == 0:
            print(f"Epoch {epoch+1} | Step {step} | Loss {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    print(f">> Rata-rata Train Loss Epoch {epoch+1}: {avg_train_loss:.4f}\n")

==== Memulai Training Seq2Seq (Summarization) ===
Epoch 1 | Step 0 | Loss 3.2938
Epoch 1 | Step 5 | Loss 2.7917
Epoch 1 | Step 10 | Loss 2.7299
>> Rata-rata Train Loss Epoch 1: 3.0352



## 7. Evaluasi Kinerja (Validation Loss & ROUGE Note)
Dalam aplikasi teks generatif seperti penerjemahan atau peringkas artikel (*Summarization*), para penulis riset standar internasional menggunakan metrik bernama **ROUGE Score** (mengukur tingkat irisan Frasa antara Teks Bangkitan vs Teks Referensi Manusia) atau **BLEU Score** (Biasa digunakan untuk terjemahan).

Namun, menghitung ROUGE Score memerlukan library pihak ketiga atau komputasi teks string yang memakan waktu. Secara dasar *PyTorch*, menghitung perbaikan performa *Validation Loss* (CrossEntropy per kata) sudah cukup membuktikan bahwa model mampu mengimitasi bahasa yang ada dalam dataset Test.

In [13]:
import math

# Simulasi set validasi
val_sample = dataset.select(range(50, 70))
tokenized_val = val_sample.map(preprocess_function, batched=True, remove_columns=dataset.column_names)
val_dataloader = DataLoader(Seq2SeqDataset(tokenized_val), batch_size=4)

model.eval()
total_val_loss = 0

with torch.no_grad():
    for batch in val_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        total_val_loss += outputs.loss.item()

avg_val_loss = total_val_loss / len(val_dataloader)
perplexity = math.exp(avg_val_loss)

print("==== Evaluasi Performa Model Seq2Seq ====")
print(f"Validation Loss : {avg_val_loss:.4f}")
print(f"Perplexity      : {perplexity:.4f}")

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

==== Evaluasi Performa Model Seq2Seq ====
Validation Loss : 3.1258
Perplexity      : 22.7781


## 8. Inferensi Hasil Akhir (Post-Training)
Pengetesan kualitas model seq2seq menggunakan `.generate` untuk menghasilkan teks ringkasan sesudah melalui parameter fine-tuning PyTorch kita di atas.

In [14]:
test_text = """
A severe storm has hit the southern coastline, causing massive power outages and flooding in several major cities. 
Authorities are urging residents to stay indoors and avoid unnecessary travel while emergency crews work to restore power and rescue those stranded by rising waters.
"""

# Jangan terlewat memasukkan instruksi "summarize: " di T5 Model
inputs = tokenizer("summarize: " + test_text.strip(), return_tensors="pt", max_length=256, truncation=True).to(device)

model.eval()
outputs = model.generate(
    **inputs,
    max_new_tokens=40,
    num_beams=4,
    early_stopping=True
)

summary = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("--- Hasil Inferensi POST-TRAIN (Seq2Seq) ---")
print("Teks Asli :", test_text.strip())
print("\nRingkasan :", summary)

--- Hasil Inferensi POST-TRAIN (Seq2Seq) ---
Teks Asli : A severe storm has hit the southern coastline, causing massive power outages and flooding in several major cities. 
Authorities are urging residents to stay indoors and avoid unnecessary travel while emergency crews work to restore power and rescue those stranded by rising waters.

Ringkasan : severe storm has hit the southern coastline, causing massive power outages and flooding in several major cities. authorities are urging residents to stay indoors and avoid unnecessary travel while emergency crews work
